# PyToch: Custom Dataset Classification

In [ ]:
import torch
import typing
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from common import CV_DATASETS_DIR

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
#
IMAGE_SIZE = (64,64)

## Prepare Datasets

In [ ]:
DATASET_PATH = CV_DATASETS_DIR/"food"/"pizza_steak_sushi"

In [ ]:
TR_DATASET_DIR = DATASET_PATH / "train"
TS_DATASET_DIR = DATASET_PATH / "test"

In [ ]:
tr_dataset = datasets.ImageFolder(root=TR_DATASET_DIR, transform=v2.Compose([
    v2.Resize(size=IMAGE_SIZE),
    v2.TrivialAugmentWide(num_magnitude_bins=31),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float, scale=True)
]))
ts_dataset = datasets.ImageFolder(root=TS_DATASET_DIR, transform=v2.Compose([
    v2.Resize(size=IMAGE_SIZE),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float, scale=True)
]))
len(tr_dataset), len(ts_dataset)

In [ ]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2, drop_last=True)
len(tr_dl), len(ts_dl)

## Define Model

In [ ]:
classes = tr_dataset.classes
print(f"Classes: {tr_dataset.classes}")

In [ ]:
class TinyVggModel(nn.Module):
    def __init__(self, in_shape: int, out_shape: int, hidden_units: int):
        super().__init__()
        self.cv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=in_shape, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.cv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*13*13,
                      out_features=out_shape),
        )

    def forward(self, inputs):
        z = self.cv_block1(inputs)
        z = self.cv_block2(z)
        outputs = self.classifier(z)
        return outputs

In [ ]:
model = TinyVggModel(in_shape=3, out_shape=len(classes), hidden_units=32).to(device)

In [ ]:
summary(model, input_size=(64, 3)+(IMAGE_SIZE))

## Training

In [ ]:
# Define loss function
loss_fn = nn.CrossEntropyLoss().to(device)

# Define metric function
accuracy_fn = MulticlassAccuracy(num_classes=len(classes)).to(device)

In [ ]:
# Define training step function
def train_step(model: nn.Module,
               loader: DataLoader,
               optimizer: optim.Optimizer,
               loss_fn: typing.Callable,
               accuracy_fn: typing.Callable,
               device: torch.device = device) -> dict:
    loss_avg, accu_avg = 0, 0
    model.train()
    for idx, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        y_logits = model(x)
        y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
        loss = loss_fn(y_logits, y)
        accu = accuracy_fn(y_pred, y)
        loss_avg += loss.item()
        accu_avg += accu.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    n_batches = len(loader)
    loss_avg /= n_batches
    accu_avg /= n_batches
    return {"loss": loss_avg, "accuracy": accu_avg}

In [ ]:
# Define testing step function
def test_step(model: nn.Module,
              loader: DataLoader,
              loss_fn: typing.Callable,
              accuracy_fn: typing.Callable,
              device: torch.device = device) -> dict:
    loss_avg, accu_avg = 0, 0
    model.eval()
    with torch.inference_mode():
        for idx, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            y_logits = model(x)
            y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
            loss = loss_fn(y_logits, y)
            accu = accuracy_fn(y_pred, y)
            loss_avg += loss.item()
            accu_avg += accu.item()
    n_batches = len(loader)
    loss_avg /= n_batches
    accu_avg /= n_batches
    return {"loss": loss_avg, "accuracy": accu_avg}

In [ ]:
N_EPOCHS = 30

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

tr_losses = []
ts_losses = []
tr_accus = []
ts_accus = []

for epoch in tqdm(range(N_EPOCHS)):
    # Do a train and test step
    results = train_step(model, tr_dl, optimizer, loss_fn, accuracy_fn, device)
    tr_loss, tr_accu = results["loss"], results["accuracy"]
    results = test_step(model, ts_dl, loss_fn, accuracy_fn, device)
    ts_loss, ts_accu = results["loss"], results["accuracy"]
    # Update LR
    scheduler.step()
    # Update history
    tr_losses.append(tr_loss)
    ts_losses.append(ts_loss)
    tr_accus.append(tr_accu)
    ts_accus.append(ts_accu)
    # Print intermediate results
    print(f"Epoch: {epoch:02} L/A: {tr_loss:.3f}/{tr_accu:3.1f} Test L/A: {ts_loss:.3f}/{ts_accu:3.1f}")

In [ ]:
epochs = range(N_EPOCHS)
plt.figure(figsize=(10, 5))
# Plot the loss
plt.subplot(1, 2, 1)
plt.plot(epochs, tr_losses, 'r-', label="Training Loss")
plt.plot(epochs, ts_losses, 'b-', label="Testing Loss")
plt.ylim(ymin=0)
plt.title("Loss")
plt.xlabel("Epochs")
plt.legend()
# Plot the accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, tr_accus, 'r-', label="Training Accuracy")
plt.plot(epochs, ts_accus, 'b-', label="Testing Accuracy")
plt.ylim(ymin=0, ymax=1.0)
plt.title("Accuracy")
plt.xlabel("Epochs")
plt.legend();

## Evaluate Model

In [ ]:
def evaluate(model: nn.Module,
             loader: DataLoader,
             loss_fn: nn.Module,
             accuracy_fn: nn.Module,
             device: torch.device = device) -> dict:
    loss_avg = 0
    accu_avg = 0
    model.eval()
    preds = []
    truth = []
    with torch.inference_mode():
        for x, y_true in tqdm(loader):
            x, y_true = x.to(device), y_true.to(device)
            y_logits = model(x)
            y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
            loss_avg += loss_fn(y_logits, y_true).item()
            accu_avg += accuracy_fn(y_pred, y_true).item()
            preds.append(y_pred)
            truth.append(y_true)
        n_batches = len(loader)
        loss_avg /= n_batches
        accu_avg /= n_batches
    return {
        "truth": torch.flatten(torch.stack(truth)).cpu(),
        "preds": torch.flatten(torch.stack(preds)).cpu(),
        "loss": loss_avg,
        "accu": accu_avg,
    }

In [ ]:
results = evaluate(model, ts_dl, loss_fn, accuracy_fn)
print(f"Loss: {results["loss"]}\nAccuracy: {results["accu"]}")

In [ ]:
metric = MulticlassConfusionMatrix(num_classes=len(classes))
metric.update(results["preds"], results["truth"])
metric.plot(labels=ts_dataset.classes);